# MiniGPT
#Creating a Transformer based LLM using GPUs and a tiny stories dataset that could make stories about a forest

###Dataset Used: https://huggingface.co/datasets/roneneldan/TinyStories

###GPU used : Nvidia T4

###Library used : JAX and its ecosystem

In [ ]:
#Author: Paarth Sharma
#FileName: tinyStories.ipynb
#ProjectName: TinyStoriesGPT
#CreationDate: 5-03-26
#ModificationDate: 5-03-26
#Description: This is a Transformer model trained on the tiny stories dataset using JAX and the XLA compiler for optimization

In [ ]:
#imports
!pip install -q grain

In [ ]:
#importing libraries
from pathlib import Path
import jax
import flax.nnx as nnx
import jax.numpy as jnp
import optax
import orbax
import tiktoken
import grain.python as pygrain

maxlen=128
num_epochs=3
tokenizer = tiktoken.get_encoding("gpt2")
tokenizer.n_vocab

In [ ]:
#steps for importing dataset for google colab
from google.colab import drive
drive.mount('/content/drive')

#first download the dataset and put it in google drive

In [ ]:
#configure file paths
DRIVE_BASE = Path("/content/drive/MyDrive/Colab Notebooks/tinyStoriesProject")

file_path = DRIVE_BASE / "data/TinyStories-train.txt"

checkpoint_dir = DRIVE_BASE / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

#Part 1: Creating the class functions for the layers of the LLM

###Creating a token embedding class

In [ ]:
class TokenEmbedding(nnx.Module):
  """
  class creates the token and the positional embedding layer of the transformer network.
  """

  #Pre : vocab size(the size of the dictionary that the model knows) , embedding size(the number of features represented by the embedding), maxlen(the maximum length/the context window the model can understand), rngs(random number generation state: used to transfer the state of the random number generation , so that all random number generators have uniformity and follow a predictable pattern)
  #Post : NA
  #Desc : assigns the token embedding and positional embedding to words for transfering them to later layers
  def __init__(self, vocab_size, embedding_size, maxlen, rngs):
    self.token_embedding = nnx.Embed(vocab_size, embedding_size, rngs = rngs)
    self.position_embedding = nnx.Embed(maxlen, embedding_size, rngs = rngs)

  #Pre : tokens
  #Post : toeken + positional embeddings
  #Desc : returns the embeddings for the model to train on
  def __call__(self, token_ids):
    sequence_length = token_ids.shape[1]
    positions = jnp.arange(sequence_length).reshape(1, sequence_length)
    return self.token_embedding(token_ids) + self.position_embedding(positions)


###Creating a causal attention mask

In [ ]:
#Pre: takes the sequence length
#Post: returns a lower_triangular_matrix masking the inputs
#Desc: masks the input token for the model to predict the future and not learn the entire dataset by looking at the future value, thus overfitting
def causal_attention_mask(sequence_length) :
  identity_matrix = jnp.ones((sequence_length, sequence_length), dtype=bool)
  lower_triangular_matrix = jnp.tril(identity_matrix)
  lower_triangular_matrix = lower_triangular_matrix[None, None, :, :]
  return lower_triangular_matrix

###Creating a transformer block

In [ ]:
class TransformerBlock(nnx.Module):
  """
  class creates a transformer block for a simple GPT model
    features :
    1. Pre Layer normalization
    2. masked multi-head self attention for previous context and training on past not future
    3. feed forward neural networks
    4. residual connection
  """

  #Pre : self, embedding_dimension, number_of_heads, rngs
  #Post : NA
  #Desc : initiate the transformer block
  def __init__ (self, embedding_dimension, number_of_heads, feed_forward_dimension, *, rngs):

    self.layer_norm_1 = nnx.LayerNorm(embedding_dimension, rngs = rngs)

    self.attention = nnx.MultiHeadAttention(
        in_features = embedding_dimension,
        qkv_features = embedding_dimension,
        out_features = embedding_dimension,
        num_heads = number_of_heads,
        rngs = rngs,
        decode = False #Running in training mode
    )

    self.layer_norm_2 = nnx.LayerNorm(embedding_dimension, rngs = rngs)

    self.feed_forward_1 = nnx.Linear(embedding_dimension, feed_forward_dimension, rngs = rngs)

    self.feed_forward_2 = nnx.Linear(feed_forward_dimension, embedding_dimension, rngs = rngs)

  #Pre : self, tokens
  #Post : attention of this layer
  #Desc : attention block with residual connections
  def __call__ (self, tokens, mask = None):

    normalized = self.layer_norm_1(tokens)

    attention = self.attention(normalized, mask = mask)

    tokens = tokens + attention #residual connection in attention for stability and gradients to propagate

    normalized_2 = self.layer_norm_2(tokens)

    hidden = self.feed_forward_1(normalized_2)

    hidden = nnx.gelu(hidden)

    hidden = self.feed_forward_2(hidden)

    tokens = tokens + hidden #second residual connection after feed forward for stability and gradients to propagate

    return tokens

###Creating a model

In [ ]:
class miniGPTModel(nnx.Module):
  """
  class creates a simple GPT model
    features :
    1. embedding layer (token and positional{using sinusoidal embeddings} )
    2. causal attention mask
    3. transformer blocks
    4. final output layer
  """

  #Pre: maxlen: int, vocab_size:int, embedding_dimension:int, number_of_heads:int, feed_forward_dimension:int, number_of_transformer_blocks:int, rngs
  #Post: NA
  #Desc: initiates the GPT model
  def __init__ (self, maxlen, vocab_size, embedding_dimension, number_of_heads, feed_forward_dimension, number_of_transformer_blocks, *, rngs):

    self.maxlen = maxlen

    self.embedding = TokenEmbedding(vocab_size, embedding_dimension, maxlen, rngs = rngs)

    self.transformer_blocks = []

    for _ in range(number_of_transformer_blocks):
      block = TransformerBlock(embedding_dimension, number_of_heads, feed_forward_dimension, rngs = rngs)
      self.transformer_blocks.append(block)

    self.final_layer_norm = nnx.LayerNorm(embedding_dimension, rngs = rngs)

    self.output_layer = nnx.Linear( embedding_dimension, vocab_size, use_bias=False, rngs=rngs)

  #Pre: self, token_ids: int
  #Post: logits:int
  #Desc: call for the model
  def __call__ (self, token_ids):

    sequence_length = token_ids.shape[1]

    mask = causal_attention_mask(sequence_length)

    embeddings = self.embedding(token_ids)

    for block in self.transformer_blocks:
      embeddings = block(embeddings, mask)

    embeddings = self.final_layer_norm(embeddings)

    logits = self.output_layer(embeddings)

    return logits

###Checking the model

In [ ]:
model = miniGPTModel(
    maxlen=128,
    vocab_size=tokenizer.n_vocab,
    embedding_dimension=256,
    number_of_heads=8,
    feed_forward_dimension=1024,
    number_of_transformer_blocks=6,
    rngs=nnx.Rngs(0)
)

In [ ]:
model

#Part 2 Loading the data using Grain

### Creating a dataset Class for efficient item getting

In [ ]:
class Dataset:

  #pre : self(object), stories(array), maxlen(int), tokenizer(object)
  #post: NA
  #desc: initializes a dataset with a maxlen of stories and a tokenizer
  def __init__ (self, stories, maxlen, tokenizer):
    self.stories = stories
    self.maxlen = maxlen
    self.tokenizer = tokenizer
    self.endtoken = tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"})[0]

  #pre : self(object)
  #post: NA
  #desc: returns the length of the stories
  def __len__(self):
    return len(self.stories)

  #pre : self(object), index(int)
  #post: tokens(int array)
  #desc: gets the item from the dataset
  def __getitem__(self, index):

    story = self.stories[index]
    tokens = self.tokenizer.encode(story, allowed_special={"<|endoftext|>"})

    #cut of the story to the maxlen
    if len(tokens) > self.maxlen:
      tokens = tokens[:self.maxlen]

    #pad if the sequence is shorter than maxlen
    tokens += [0] * (self.maxlen - len(tokens))

    return tokens



### Creating a dataloading function

In [ ]:
#pre : filepath(string), batch_size(int),
#post: dataLoader , estimated batches per epoch
#desc: load and process the dataset using a filepath and then output a dataloader for that dataset
def load_and_process_dataset(
    file_path ,
    batch_size ,
    maxlen ,
    max_stories ,
    num_epochs ,
    shuffle ,
    seed = 60
):

  #create a array for the loading file in chunks
  stories = []
  current_story = []

  #open the file
  with open(file_path, "r", encoding='utf-8') as file:

    #load the text file line by line
    for line in file:

      #if this line has the end of a story
      if "<|endoftext|>" in line:

        #split the two parts
        parts = line.split('<|endoftext|>')

        #join the last part of the story to form a story
        for i, part in enumerate(parts[:-1]):

          # add the last part of the story to the array
          current_story.append(part)

          #form the story text
          story_text = "".join(current_story).strip()

          #if there is a story text then append it into a story in the stories array
          if story_text:
            stories.append(story_text + '<|endoftext|>' )

            #if the length of maximum stories is reached then exit
            if len(stories) >= max_stories:
               break
          #reset the current story to a empty array
          current_story = []

        # Last part becomes start of next story
        if parts[-1].strip():
          current_story = [parts[-1]]

        #exit if the maximum length is reached
        if len(stories) >= max_stories:
          break

      #append the line to the current story
      else:
        current_story.append(line)

  #estimate the batches
  estimated_batches_per_epoch = len(stories) // batch_size

  # Create efficient dataset
  dataset = Dataset(stories, maxlen, tokenizer)

  # Configure sampler with sharding support
  sampler = pygrain.IndexSampler(
    num_records=len(dataset),
    shuffle=shuffle,
    seed=seed,
    shard_options=pygrain.NoSharding(),
    num_epochs=num_epochs,
  )

  # Create DataLoader with efficient batching
  dataloader = pygrain.DataLoader(
    data_source=dataset,
    sampler=sampler,
    operations=[
      pygrain.Batch(batch_size=batch_size, drop_remainder=True)
    ]
  )

  return dataloader, estimated_batches_per_epoch


# Part 3 Training and saving the model

### loading the Data

In [ ]:
data_loader, estimated_batches_per_epoch = load_and_process_dataset(
    file_path=file_path,
    batch_size=32,
    maxlen=maxlen,
    max_stories=100000,
    num_epochs=3,
    shuffle=True,
    seed = 60
)

###Defining the loss function for the model

In [ ]:
def loss_function(model, batch):
  input, target = batch
  logits = model(input)

  loss = optax.softmax_cross_entropy_with_integer_labels(
        logits, target
  ).mean()

  return loss, logits

###Warmup steps for warming up the parameters

In [ ]:
num_epochs=3
total_steps = estimated_batches_per_epoch * num_epochs
warmup_steps = max(1, total_steps // 10)

warmup_schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0,
    peak_value=3e-4,
    warmup_steps=warmup_steps,
    decay_steps=total_steps,
    end_value=1e-5
)

###Defining the metrics for the model

In [ ]:
metrics = nnx.MultiMetric(
    loss=nnx.metrics.Average('loss'),
)

###Defining the optimizer for the model

In [ ]:
optimizer = nnx.Optimizer(
    model,
    optax.adamw(learning_rate=warmup_schedule, weight_decay=0.01),
    wrt=nnx.Param
)

###Defining the training step and Jit compiling it

In [ ]:
@nnx.jit
def train_step(model, optimizer, metrics, batch):
    grad_function = nnx.value_and_grad(loss_function, has_aux=True)
    (loss, logits), grads = grad_function(model, batch)

    metrics.update(loss=loss)
    optimizer.update(model, grads)

###Running the training steps

In [ ]:
checkpointer = orbax.checkpoint.PyTreeCheckpointer()

metrics_history = {'train_loss': []}

prep_target_batch = jax.vmap(
    lambda tokens: jnp.concatenate((tokens[1:], jnp.array([0]))))

checkpoint_every_n_steps = 100  # Save every 100 steps

for epoch in range(num_epochs):

    step = 0

    for batch in data_loader:

        input_batch = jnp.array(jnp.array(batch).T).astype(jnp.int32)
        target_batch = prep_target_batch(
            jnp.array(jnp.array(batch).T)).astype(jnp.int32)

        print(".", end="")

        train_step(model, optimizer, metrics, (input_batch, target_batch))

        if (step + 1) % 2 == 0:

            for metric, value in metrics.compute().items():
                metrics_history[f'train_{metric}'].append(value)

            metrics.reset()

            current_lr = warmup_schedule(step)

            print(f"\nEpoch: {epoch + 1}, Step {step + 1}, Loss: {metrics_history['train_loss'][-1]:.4f}, "
                  f"LR: {current_lr:.2e}")

        # ---- Checkpointing ----
        if (step + 1) % checkpoint_every_n_steps == 0:
            checkpoint_path = checkpoint_dir / f"epoch_{epoch+1}_step_{step+1}"
            checkpointer.save(checkpoint_path, nnx.state(model), force=True)
            print(f"\nCheckpoint saved at {checkpoint_path}")

        step += 1

    # Optional: save at end of every epoch too
    epoch_checkpoint_path = checkpoint_dir / f"epoch_{epoch+1}_final"
    checkpointer.save(epoch_checkpoint_path, nnx.state(model), force=True)
    print(f"\nEpoch {epoch+1} checkpoint saved at {epoch_checkpoint_path}")

### Testing the output

In [ ]:
checkpointer = orbax.checkpoint.PyTreeCheckpointer()
restored_state = checkpointer.restore(checkpoint_dir / "epoch_1_final")

# Use merge instead of update
graphdef, _ = nnx.split(model)
model = nnx.merge(graphdef, restored_state)

In [ ]:
def generate(model, prompt, max_new_tokens=120, temperature=0.8, top_k=40):
    tokens = tokenizer.encode(prompt)
    rng = jax.random.PRNGKey(42)
    end_token = tokenizer.encode(
        "<|endoftext|>", allowed_special={"<|endoftext|>"}
    )[0]

    for _ in range(max_new_tokens):
        padded = tokens[-maxlen:]
        actual_len = len(padded)
        pad_len = maxlen - actual_len

        # Stop before hitting context window limit
        if actual_len >= maxlen - 5:
            break

        input_ids = jnp.array(padded + [0] * pad_len).reshape(1, -1)
        logits = model(input_ids)[0, actual_len - 1, :]

        # Top-k filtering
        top_vals, top_idx = jax.lax.top_k(logits, top_k)
        logits = jnp.full_like(logits, -jnp.inf).at[top_idx].set(top_vals)

        probs = jax.nn.softmax(logits / temperature)
        rng, subkey = jax.random.split(rng)
        next_token = int(jax.random.choice(subkey, len(probs), p=probs))

        tokens.append(next_token)

        # Stop at end token
        if next_token == end_token:
            break

        # Stop if stuck in repetition loop
        if len(tokens) >= 5 and len(set(tokens[-5:])) == 1:
            break

    # Fix all escape sequences from dataset
    output = tokenizer.decode(tokens)
    output = (output
        .replace("\\n", "\n")
        .replace("\\t", "\t")
        .replace("\\'", "'")
        .replace('\\"', '"')
        .replace("\\\\", "\\")
        .replace("\n", "")
    )
    return output


def generate_story(model, story_prompt, temperature, max_new_tokens):
    return generate(model, story_prompt,
                    max_new_tokens=max_new_tokens,
                    temperature=temperature)

In [ ]:
generate_story(model, "The fox was in the forest with the pigs", 0.8, 100)

In [ ]:
from safetensors.numpy import save_file, load_file
import numpy as np
import jax
import flax.nnx as nnx

# Restore BEST checkpoint (epoch 3)
checkpointer = orbax.checkpoint.PyTreeCheckpointer()
restored_state = checkpointer.restore(checkpoint_dir / "epoch_1_final")
graphdef, _ = nnx.split(model)
model = nnx.merge(graphdef, restored_state)

# Extract state
_, state = nnx.split(model)

# Convert to flat dict
flat_arrays = {}
leaves, treedef = jax.tree_util.tree_flatten_with_path(state)

for path, leaf in leaves:
    key = ".".join([
        str(p.key) if hasattr(p, 'key')
        else str(p.idx) if hasattr(p, 'idx')
        else str(p)
        for p in path
    ])
    flat_arrays[key] = np.array(leaf)

print(f"Number of tensors: {len(flat_arrays)}")
print("Sample keys:", list(flat_arrays.keys())[:5])

# Save to Google Drive so it persists
save_path = DRIVE_BASE / "minigpt_weights.safetensors"
save_file(flat_arrays, save_path)
print(f"Saved to {save_path}")

# Verify it loads back correctly
loaded = load_file(save_path)
assert set(loaded.keys()) == set(flat_arrays.keys()), "Key mismatch!"
print(f"Verified — {len(loaded)} tensors loaded successfully")
print(f"File size: {save_path.stat().st_size / 1024 / 1024:.2f} MB")

In [ ]:
model_path = "/content/drive/MyDrive/Colab Notebooks/tinyStoriesProject/serving/minigpt_weights.safetensors"

flat_arrays = load_file(model_path)

# Print both to compare
graphdef, state = nnx.split(model)
leaves, _ = jax.tree_util.tree_flatten_with_path(state)

model_keys = []
for path, _ in leaves:
    key = ".".join([
        str(p.key) if hasattr(p, 'key')
        else str(p.idx) if hasattr(p, 'idx')
        else str(p)
        for p in path
    ])
    model_keys.append(key)

print("MODEL KEYS:", model_keys[:10])
print("SAFETENSOR KEYS:", list(flat_arrays.keys())[:10])